# 1. Explore and validate field data

This notebook is the first reproducible checkpoint. It parses the source surveys, checks the normalized table, and builds the demand tensor used by the game model. Reusable logic remains in `src/`, not in this notebook.

In [ ]:
from pathlib import Path
import os
import pandas as pd

# Make the notebook reliable whether launched from the root or notebooks/.
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
os.chdir(root)

from src.config import NODES, ROUTES, WINDOWS
from src.data_parser import build_boarding_counts
from src.demand import build_demand_tensor, load_field_sheets
from src.paths import ProjectPaths

paths = ProjectPaths.discover()
paths.raw_data, paths.processed_data

In [ ]:
raw_files = sorted(paths.raw_data.glob('DATA-*.csv'))
pd.DataFrame({'source_file': [file.name for file in raw_files]})

In [ ]:
build_boarding_counts(paths.raw_data, paths.processed_data)
field_data = load_field_sheets(paths.processed_data)
field_data.head()

## Coverage checks

Unexpected missing nodes, routes, or time windows are a reason to inspect the raw survey files before running the model. A zero in the demand tensor means no matching recorded observations.

In [ ]:
coverage = (
    field_data.groupby(['node', 'route', 'window'], observed=True)
    .agg(observations=('date', 'size'), boardings=('passengers_boarding', 'sum'))
    .reset_index()
)
coverage.sort_values(['node', 'route', 'window']).head(12)

In [ ]:
demand = build_demand_tensor(field_data)
route_demand = pd.Series(demand.sum(axis=(0, 2)), index=ROUTES, name='passengers_per_window')
route_demand.to_frame()